In [1]:
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

def load_data(path, subset='training'):
    """
    Carga los datos y prepara el generador de imágenes con normalización.
    """
    labels = pd.read_csv(path + 'labels.csv')
    
    # Creamos el generador. Definimos un split de validación del 25%
    # y normalizamos los píxeles (dividiendo por 255).
    data_datagen = ImageDataGenerator(
        validation_split=0.25, 
        rescale=1./255
    ) 
    
    data_gen_flow = data_datagen.flow_from_dataframe(
        dataframe=labels,
        directory=path + 'final_files/',
        x_col='file_name',
        y_col='real_age',
        target_size=(224, 224),
        batch_size=16,
        class_mode='raw', # Usamos 'raw' porque es una tarea de regresión (números)
        subset=subset,
        seed=12345)

    return data_gen_flow

def create_model(input_shape):
    """
    Define la arquitectura del modelo usando Transfer Learning con ResNet50.
    """
    # Cargamos ResNet50 pre-entrenada con ImageNet, sin la "cabeza" (include_top=False)
    backbone = ResNet50(weights='imagenet', 
                        input_shape=input_shape,
                        include_top=False)

    model = Sequential()
    model.add(backbone)
    # GlobalAveragePooling ayuda a reducir la dimensionalidad antes de la capa final
    model.add(GlobalAveragePooling2D())
    # Una sola neurona de salida con 'relu' porque la edad siempre es positiva
    model.add(Dense(1, activation='relu'))

    # Usamos un learning rate bajo para no "arruinar" el conocimiento previo de ResNet
    optimizer = Adam(learning_rate=0.0001) 
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

    return model

def train_model(model, train_data, test_data, batch_size=None, epochs=3):
    """
    Ejecuta el entrenamiento del modelo.
    """
    model.fit(
        train_data,
        validation_data=test_data,
        batch_size=batch_size,
        epochs=epochs,
        steps_per_epoch=len(train_data),
        validation_steps=len(test_data),
        verbose=2
    )

    return model

# --- Ejecución del Flujo ---

input_shape = (224, 224, 3)
# NOTA: Asegúrate de que estas rutas coincidan con donde tienes tus datos localmente
path = '/datasets/faces/' 

train = load_data(path, subset='training')
test = load_data(path, subset='validation') # Usamos 'validation' para el conjunto de prueba

model = create_model(input_shape)
model = train_model(model, train, test, epochs=5) # 5 épocas suele ser mejor para bajar el MAE

FileNotFoundError: [Errno 2] No such file or directory: '/datasets/faces/labels.csv'